In [6]:
import numpy as np
import pandas as pd
import polars as pl
from factor_analysis.utils import AlphaUtils
from factor_analysis.plotter import FactorPlotter
from factor_analysis.config import AlphaConfig
from IPython.display import HTML, display

# 1. 造数据（完全使用 Polars，不需要 pyarrow）
rng = np.random.default_rng(42)
n_inst, n_day = 50, 100
instruments = [f"{i:06d}.SZ" for i in range(n_inst)]
dates = pl.date_range(start=pl.date(2024, 1, 1), end=pl.date(2024, 5, 20), interval="1d", eager=True)[:n_day]

df = (
    pl.DataFrame({"date": dates})
    .join(pl.DataFrame({"instrument": instruments}), how="cross")
    .with_columns([
        pl.Series("factor", rng.normal(size=n_inst * n_day)),
        pl.Series("daily_ret", rng.normal(0, 0.02, size=n_inst * n_day))
    ])
)

# 2. 算 IC
ic = AlphaUtils.spearman_ic_by_date(df)

# 3. 算分组累积收益
group_num = 5
gdf = df.with_columns(
    (pl.col("factor").rank(method="ordinal").over("date") / pl.len().over("date")).alias("pct")
).with_columns(
    pl.min_horizontal((pl.col("pct") * group_num).cast(pl.Int32), pl.lit(group_num - 1)).cast(pl.Utf8).alias("group")
).drop("pct")

groupret = gdf.group_by(["date", "group"]).agg(pl.col("daily_ret").mean().alias("g_ret"))
pivot = groupret.pivot(values="g_ret", index="date", on="group").sort("date")
pivot = pivot.with_columns(pl.all().exclude("date").shift(1)).drop_nulls()

# 4. 准备画图数据
ic_plot_df = ic.with_columns([
    pl.col('g_ic').cum_sum().alias('ic_cumsum'),
    pl.col('g_ic').rolling_mean(22).alias('ic_roll_ma')
]).to_pandas()

cumret_plot_df = pivot.to_pandas().set_index('date')
cumret_plot_df['bm'] = 0.0 
for col in cumret_plot_df.columns:
    if col != 'bm': cumret_plot_df[col] = cumret_plot_df[col].cumsum()

# 5. 画图并展示
fig_ic = FactorPlotter.bar_chart(ic_plot_df, "每日 IC 与 22日滚动均值", x='date', y=['g_ic', 'ic_roll_ma'])
fig_cum = FactorPlotter.cumret_chart(cumret_plot_df, "分组累积收益曲线 (单利)")

display(HTML(f"<style>{AlphaConfig.REPORT_CSS}</style><div class='factor-report'>"))
display(HTML("<h2 class='chart-title'>IC 曲线</h2>"))
display(fig_ic)
display(HTML("<h2 class='chart-title'>分组累积收益曲线</h2>"))
display(fig_cum)

In [7]:
import numpy as np
import pandas as pd
import polars as pl
from types import SimpleNamespace
from factor_analysis.report import ReportBuilder
from factor_analysis.plotter import FactorPlotter

# ======================================================================
# 1. 制造假数据，并直接构造一个"假的" FactorAnalyzer 实例 (fa)
# ======================================================================
rng = np.random.default_rng(42)
n_inst, n_day = 100, 250
instruments = [f"{i:06d}.SZ" for i in range(n_inst)]
dates = pl.date_range(start=pl.date(2023, 1, 1), end=pl.date(2024, 12, 31), interval="1d", eager=True)[:n_day]

df = (
    pl.DataFrame({"date": dates})
    .join(pl.DataFrame({"instrument": instruments}), how="cross")
    .with_columns([
        pl.Series("factor", rng.normal(size=n_inst * n_day)),
        pl.Series("daily_ret", rng.normal(0.0005, 0.02, size=n_inst * n_day))
    ])
)

# 计算分组累积收益
group_num = 5
gdf = df.with_columns(
    (pl.col("factor").rank(method="ordinal").over("date") / pl.len().over("date")).alias("pct")
).with_columns(
    pl.min_horizontal((pl.col("pct") * group_num).cast(pl.Int32), pl.lit(group_num - 1)).cast(pl.Utf8).alias("group")
).drop("pct")

groupret = gdf.group_by(["date", "group"]).agg(pl.col("daily_ret").mean().alias("g_ret"))
pivot = groupret.pivot(values="g_ret", index="date", on="group").sort("date")
# 补齐缺失列（如果有的话）
for i in range(group_num):
    if str(i) not in pivot.columns:
        pivot = pivot.with_columns(pl.lit(0.0).alias(str(i)))
pivot = pivot.with_columns((pl.col("4") - pl.col("0")).alias("ls"))
pivot = pivot.with_columns(pl.lit(0.0).alias("bm"))

# 计算 IC 数据
from factor_analysis.utils import AlphaUtils
ic_df = AlphaUtils.spearman_ic_by_date(df).with_columns([
    pl.col('g_ic').cum_sum().alias('ic_cumsum'),
    pl.col('g_ic').rolling_mean(22).alias('ic_roll_ma')
])

# 构造一个假的 fa 实例
fa = SimpleNamespace()
fa.params = {'factor_field': '测试因子', 'instruments': '全市场', 'group_num': 5, 'factor_direction': 1, 'benchmark': '无'}
fa.start_date = "2023-01-01"
fa.end_date = "2024-12-31"
fa.TOP_N_INS = 5

# 将数据存入 fa
fa.group_cumret = pivot
fa.ic_df = ic_df
fa.factor_data = df

# 伪造表格数据 (简单 mock 一下 whole_perf 和 yearly_perf)
fa.whole_perf = pl.DataFrame({
    'portfolio': ['long', 'short', 'long_short'],
    'ic': [0.05, -0.02, 0.07], 'ir': [0.4, -0.1, 0.5], 'turnover': [0.1, 0.2, 0.3],
    'return_ratio': [0.15, -0.05, 0.20], 'annual_return_ratio': [0.10, -0.03, 0.13],
    'ex_return_ratio': [0.12, -0.08, 0.20], 'ex_annual_return_ratio': [0.08, -0.05, 0.13],
    'sharp_ratio': [1.2, -0.5, 1.5], 'return_volatility': [0.1, 0.15, 0.1],
    'information_ratio': [0.8, -0.3, 1.0], 'max_drawdown': [-0.05, -0.15, -0.04],
    'win_percent': [0.55, 0.45, 0.60], 'ic_252': [0.04, -0.01, 0.05], 'ret_252': [0.10, -0.02, 0.12]
})
fa.yearly_perf = pl.DataFrame({
    'year': ['2023', '2024'], 'ic': [0.04, 0.06], 'return_ratio': [0.08, 0.07],
    'annual_return_ratio': [0.08, 0.07], 'ex_return_ratio': [0.06, 0.05], 'ex_annual_return_ratio': [0.06, 0.05],
    'sharp_ratio': [1.1, 1.3], 'return_volatility': [0.1, 0.09], 'max_drawdown': [-0.04, -0.03],
    'win_percent': [0.54, 0.56], 'trading_days': [242, 242]
})

# ======================================================================
# 2. 调用 ReportBuilder 保存 HTML
# ======================================================================
# output_html 指定文件名，show=False 表示只保存文件不在 Jupyter 里显示
rb = ReportBuilder(fa)
rb.render(output_html='factor_report.html', show=False)

print("✅ HTML 报告已成功保存至当前目录下的 factor_report.html")

✅ HTML 报告已成功保存至当前目录下的 factor_report.html
